# Course notebook — Multimodal and Frontier Systems

Understand multimodal pipelines and evaluate emerging AI capabilities through enduring principles and evidence.

Use Python 3.10 or later. All inputs are included and the workbook runs without API keys. Each lecture demonstration is a small model of the lecture mechanism. Run it, compare the result with your prediction, and use the section checkpoint to record what you learned.


## Practice case · The records that weren’t written for a model

A public archive lets researchers search handwritten forms, meeting recordings, and plain-text transcripts. Elise, the archivist, notices that one exhibit is overrepresented because its pages scan cleanly and its speakers use standard studio audio. The team wants useful discovery while letting users see where evidence is uncertain or incomplete.

The workplace scenario is fictional.


## Documents, vision, and audio


### Lecture 01 · Vision and Document Intelligence

Understand image representations, vision-language models, OCR, layout, tables, charts, spatial relationships, and provenance.

#### Two lines can become one wrong record

A faded form places a name above its address. The vision system reads the lines correctly but attaches the address to the neighboring record.

Evaluate field extraction and record association separately. Preserve page and region references so reviewers can inspect low-confidence mappings rather than only correcting text.

#### Read the mechanism

**vision encoders**

Vision encoders map images to embeddings or tokens for multimodal models—ViT, CLIP-style architectures.

Chart screenshot encoded to tokens fused with text question about Q3 revenue trend.

**OCR**

OCR extracts text from scanned images and photos, introducing recognition errors that propagate to chunks and answers. Confidence scores help gate low-quality extractions.

Scanned contracts with skewed pages need deskew preprocessing before OCR.

**layout models**

Layout models detect reading order, tables, figures, and headings in documents beyond raw OCR boxes.

Invoice layout model separates line items table from footer terms for field extraction.

**document AI**

Document AI pipelines combine OCR, layout, extraction, and validation for structured data from unstructured files.

Extract vendor, line items, tax from PDF invoices into ERP JSON with confidence scores.

**visual grounding**

Visual grounding links language to regions or objects in images—pointing, bounding boxes, UI elements.

Model clicks 'Submit' button coordinates in screenshot for computer-use agent.

#### Worked example

**Situation:** A document system must combine tables, charts, and text without losing source provenance for compliance audits.

**Baseline:** OCR plain text dump—table cells merge, chart values lost.

**Application:** Pipeline with layout detection, OCR confidence per block, table structure recovery, vision-language model for chart reading, store bounding boxes and page IDs with extracted fields.

**Test cases:** (1) Normal: digital PDF table. (2) Boundary: skewed scan 80% OCR confidence. (3) Adversarial: chart image with misleading axis scale.

**Measurement:** Field-level F1, page-level citation accuracy, provenance completeness.

**Design question:** When must human review gate fields below OCR confidence threshold?

**Predict before running.** Preserve spatial structure and provenance when converting visual documents into model context. Identify the input you would hold fixed and the observation that would change your decision.


In [ ]:
"""Lab 13.1: Vision and Document Intelligence"""

CHAPTER = "13.1"
print("chapter hook:", CHAPTER)
blocks = [
    {"type": "table", "text": "PTO cap 240", "page": 3, "confidence": 0.96},
    {"type": "chart", "text": "headcount trend", "page": 4, "confidence": 0.71},
]
THRESH = 0.85
for b in blocks:
    print(b["type"], "auto_extract:", b["confidence"] >= THRESH)
print("---")
print("change one input above, predict output, re-run")


#### Interpret and apply

Extract fields from documents and evaluate field and page-level accuracy.

Change one input, record the prediction before running again, and compare with the unchanged output. If a result disappoints, record which intermediate stage first diverged.

1. Why preserve spatial structure in document AI?
2. How do layout models change RAG chunk quality?
3. What document baseline is plain OCR text only?

<details><summary>Check your reasoning</summary>

Q1: Citations need page/bbox; tables need cell structure. Q2: Chunks align to semantic blocks not arbitrary splits. Q3: strip-all-layout text file.

</details>


### Lecture 02 · Speech and Audio

Connect ASR, diarization, audio understanding, TTS, streaming, latency, consent, and voice safety.

#### A transcript can hide where the audio failed

The system transcribes the chair well but breaks down on remote participants who have noisy connections. A smooth transcript misattributes one decision.

Slice by speaker and recording condition, and check whether meaning survived rather than only counting word errors. Keep a link to the source time range for a reviewer.

#### Read the mechanism

**speech recognition**

Speech recognition (ASR) transcribes audio to text with word error rate varying by accent, noise, and domain.

Call center ASR feeds ticket summary pipeline with custom vocabulary for product names.

**diarization**

Diarization labels who spoke when in multi-speaker audio—essential for meetings and support calls.

Support call transcript tags agent versus customer utterances for QA scoring.

**text-to-speech**

Text-to-speech synthesizes natural audio from text with voice, prosody, and latency trade-offs.

IVR reads dynamic account balance with consistent brand voice under 500ms first byte.

**streaming audio**

Streaming audio processes speech incrementally for real-time captions and voice agents.

Live meeting captions display partial hypotheses updated as speaker continues.

**voice safety**

Voice safety covers consent, voice cloning abuse, deepfake detection, and secure storage of biometric voice data.

Require explicit opt-in before cloning executive voice for IVR.

#### Worked example

**Situation:** Onboarding includes live HR briefing recordings; system must transcript, diarize speakers, and flag low-confidence spans for review.

**Baseline:** Batch ASR with no timestamps—cannot align to policy mentions.

**Application:** Streaming ASR simulation with chunk timestamps, speaker diarization labels, confidence-based review queue, consent logging for voice data retention.

**Test cases:** (1) Normal: clean single speaker. (2) Boundary: overlapping crosstalk. (3) Adversarial: synthetic voice command injection in recording.

**Measurement:** WER, diarization error rate, p95 streaming latency, flagged-span review rate.

**Design question:** What latency budget forces streaming vs batch ASR architecture?

**Predict before running.** Audio systems are temporal, identity-sensitive, and latency-constrained. Identify the input you would hold fixed and the observation that would change your decision.


In [ ]:
"""Lab 13.2: Speech and Audio"""

CHAPTER = "13.2"
print("chapter hook:", CHAPTER)
segments = [
    {"start": 0.0, "end": 2.5, "speaker": "HR", "text": "welcome", "conf": 0.95},
    {"start": 2.5, "end": 4.0, "speaker": "?", "text": "mumbled", "conf": 0.55},
]
for s in segments:
    flag = s["conf"] < 0.75
    print(s, "review:", flag)
print("---")
print("change one input above, predict output, re-run")


#### Interpret and apply

Build a transcript pipeline with timestamps and confidence handling.

Change one input, record the prediction before running again, and compare with the unchanged output. If a result disappoints, record which intermediate stage first diverged.

1. Why are audio systems latency- and identity-sensitive?
2. How should low-confidence spans surface in UX?
3. What speech baseline is batch-only with no diarization?

<details><summary>Check your reasoning</summary>

Q1: Real-time use needs streaming; speakers matter for attribution. Q2: Highlight for human verify before acting. Q3: Whole-file transcript paragraph.

</details>


**Section checkpoint.** Annotate one extracted document field with a page reference and one speech result with a time range.


## Generative media and bounded action


### Lecture 03 · Image and Video Generation

Study diffusion, conditioning, latent representations, control, evaluation, provenance, copyright, and content safety.

#### Generated visuals need provenance and editorial choice

The team creates a period-style reconstruction for a public exhibit. A reader mistakes it for an archival photograph.

Record and disclose provenance at publication. Evaluate whether the reconstruction supports the intended explanation and whether labeling remains visible when the image is shared or cropped.

#### Read the mechanism

**diffusion**

Diffusion models generate images by iterative denoising from noise, conditioning on text or layout.

Marketing generates product hero images from prompt plus brand color conditioning.

**conditioning**

Conditioning steers generation with extra inputs—text, masks, ControlNet edges, brand assets.

Logo placement conditioned via layout mask keeps brand mark in safe zone.

**latent space**

Latent space in generative models compresses images to lower-dimensional representations for efficient editing and generation.

Latent diffusion edits background without re-encoding full resolution each step.

**video generation**

Video generation extends image models temporally—short clips from text with consistency and motion challenges.

Generate 5s product demo clip from storyboard prompts for social ads.

**provenance**

Provenance for generated media records model, prompt, timestamp, and user for copyright and authenticity disputes.

C2PA metadata embeds creation tool and prompt hash in exported campaign image.

#### Worked example

**Situation:** Marketing wants AI-generated onboarding welcome videos; brand and legal need controllable, traceable assets.

**Baseline:** Generate clips from prompt only—inconsistent characters and no provenance.

**Application:** Design diffusion workflow with conditioning (logo palette, script), latent-space edit controls, rubric evaluating consistency/safety/provenance, watermark/metadata policy.

**Test cases:** (1) Normal: short clip matching storyboard. (2) Boundary: minor character drift frame-to-frame. (3) Adversarial: prompt attempting copyrighted character likeness.

**Measurement:** Rubric pass rate, frame consistency score, provenance metadata presence.

**Design question:** Which rubric dimension gates release before aesthetic quality?

**Predict before running.** Generative quality includes controllability, consistency, provenance, safety, and workflow fit. Identify the input you would hold fixed and the observation that would change your decision.


In [ ]:
"""Lab 13.3: Image and Video Generation"""

CHAPTER = "13.3"
print("chapter hook:", CHAPTER)
rubric = {"brand_match": 0.9, "safety": 1.0, "provenance": 1.0, "aesthetic": 0.85}
gates = {"safety": 1.0, "provenance": 1.0}
release = all(rubric[k] >= gates[k] for k in gates)
print({"release": release, "scores": rubric})
print("---")
print("change one input above, predict output, re-run")


#### Interpret and apply

Design an evaluation rubric for generated campaign assets.

Change one input, record the prediction before running again, and compare with the unchanged output. If a result disappoints, record which intermediate stage first diverged.

1. Why does generative quality include provenance and safety?
2. How does conditioning differ from post-hoc editing?
3. What generation baseline evaluates prettiness only?

<details><summary>Check your reasoning</summary>

Q1: Legal and trust require traceability and misuse controls. Q2: Conditioning constrains generation upfront. Q3: Human likes clip with no metadata policy.

</details>


### Lecture 04 · Computer Use and Embodied Action

Model perception–action loops, UI grounding, coordinate and semantic actions, recovery, permissions, and physical-world constraints.

#### A visible button can still change a real record

An experimental tool proposes editing an archive entry after spotting an apparent typo. The field may contain the historical spelling that curators need to keep.

Make inspection read-only by default. Put the proposed change and supporting evidence in front of an authorized reviewer. Check recoverability and expected state before applying an approved edit.

#### Read the mechanism

**computer use**

Computer use agents perceive screens and emit mouse/keyboard actions to complete software tasks.

Agent fills expense form in internal web app from receipt image with confirmation gates.

**visual grounding**

Visual grounding links language to regions or objects in images—pointing, bounding boxes, UI elements.

Model clicks 'Submit' button coordinates in screenshot for computer-use agent.

**action spaces**

Action spaces define allowed agent operations—click, type, scroll, API call—with granularity affecting reliability.

Semantic actions ('open_settings') beat raw coordinates when UI reskins change layout.

**recovery**

Recovery restores consistent state after crashes, tool failures, or partial commits. It requires durable checkpoints and compensating actions.

After payment timeout, recovery verifies ledger state before retry or refund.

**robotics interfaces**

Robotics interfaces connect AI planners to sensors and actuators with safety interlocks and real-time constraints.

Warehouse robot API accepts move commands only within geofenced zones with E-stop.

#### Worked example

**Situation:** Prototype automates browser-based benefits enrollment; wrong click is hard to reverse.

**Baseline:** Agent clicks immediately from model coordinates—misclicks enroll wrong plan.

**Application:** Design computer-use loop with UI grounding, semantic action targets (button labels not raw x,y), confirmation for irreversible steps, recovery path on layout change.

**Test cases:** (1) Normal: stable form fill. (2) Boundary: dynamic DOM reload mid-task. (3) Adversarial: deceptive button labels ("No" means confirm).

**Measurement:** Task success, misclick rate, recovery success, human confirmation bypass attempts.

**Design question:** When are coordinate actions unacceptable compared to semantic targeting?

**Predict before running.** Acting through interfaces adds uncertainty and irreversible side effects to ordinary agent loops. Identify the input you would hold fixed and the observation that would change your decision.


In [ ]:
"""Lab 13.4: Computer Use and Embodied Action"""

CHAPTER = "13.4"
print("chapter hook:", CHAPTER)
actions = [
    {"type": "click_semantic", "target": "Submit enrollment", "risk": "high"},
    {"type": "click_xy", "x": 120, "y": 400, "risk": "high"},
]
for a in actions:
    needs_confirm = a["risk"] == "high"
    print(a["type"], "confirm:", needs_confirm)
print("---")
print("change one input above, predict output, re-run")


#### Interpret and apply

Design a safe browser task with confirmation and recovery.

Change one input, record the prediction before running again, and compare with the unchanged output. If a result disappoints, record which intermediate stage first diverged.

1. Why do interface actions add irreversible risk?
2. How does recovery work when UI layout shifts?
3. What computer-use baseline clicks without confirmation?

<details><summary>Check your reasoning</summary>

Q1: UI actions change real account state. Q2: Re-ground elements, rewind to last checkpoint. Q3: Raw coordinate agent on production HR portal.

</details>


**Section checkpoint.** Define one review gate for a generated asset and one approval condition for an irreversible action.


## Emerging systems and evidence


### Lecture 05 · Long Context, World Models, and Continual Learning

Examine active directions without mistaking larger demonstrations for solved engineering problems.

#### A longer context window can hide missing retrieval

A model receives a long transcript but overlooks a brief correction two-thirds of the way through. Flipping to a bigger context window raises cost without fixing the omission.

Place a known correction at different positions and measure whether the answer uses it. Compare targeted retrieval with the longer window using the same corrections and authority decisions.

#### Read the mechanism

**long context**

Long context models attend to hundred-thousand-plus tokens in one window—reducing need for retrieval but not eliminating cost or lost-in-middle effects.

Pasting entire contract for QA works until cost and middle-section attention degrade answers.

**world models**

World models learn predictive representations of environments for planning or simulation—active research area with engineering gaps.

Game agent predicts next frame state to plan moves without full environment queries.

**continual learning**

Continual learning updates models on new data without catastrophic forgetting of prior tasks. Production systems often prefer explicit versioning and retraining over true continual learning today.

Adding new product SKUs to classifier without retraining on old SKUs should not collapse accuracy on legacy labels.

**memory**

Memory in frontier systems spans working, episodic, and semantic stores beyond context windows—implementation varies widely.

Agent stores user preferences in durable memory retrieved each session.

**test-time adaptation**

Test-time adaptation updates model behavior during inference from recent inputs—risky for stability without guardrails.

Adapter adjusts to user's jargon mid-session if enabled with rollback.

#### Worked example

**Situation:** Vendor claims 1M-token context replaces retrieval for policy assistant; architect must evaluate against strong baselines.

**Baseline:** Accept vendor demo as proof—no controlled comparison.

**Application:** Decompose claim into representation, memory, search, learning components; compare long-context vs RAG vs explicit state on cost, accuracy, freshness for policy QA slice.

**Test cases:** (1) Normal: answer in first 10k tokens. (2) Boundary: needle buried at 800k. (3) Adversarial: policy updated after context cached.

**Measurement:** Accuracy vs position, cost per query, freshness lag on updated clause.

**Design question:** Which failure mode proves long context did not solve retrieval?

**Predict before running.** Frontier techniques should be decomposed into representation, memory, search, learning, and control claims. Identify the input you would hold fixed and the observation that would change your decision.


In [ ]:
"""Lab 13.5: Long Context, World Models, and Continual Learning"""

CHAPTER = "13.5"
print("chapter hook:", CHAPTER)
methods = {"long_context": 0.88, "rag": 0.91, "explicit_state": 0.89}
cost = {"long_context": 9, "rag": 3, "explicit_state": 2}
print({m: {"acc": methods[m], "cost": cost[m]} for m in methods})
print("inspect step", 1)
print("---")
print("change one input above, predict output, re-run")


#### Interpret and apply

Compare a frontier method with retrieval, explicit state, or fine-tuning baselines.

Change one input, record the prediction before running again, and compare with the unchanged output. If a result disappoints, record which intermediate stage first diverged.

1. Why decompose frontier claims into engineering components?
2. When might long context still lose to RAG?
3. What frontier baseline trusts demos?

<details><summary>Check your reasoning</summary>

Q1: Separates hype from testable mechanisms. Q2: Freshness, cost, needle depth, authorization per chunk. Q3: Vendor keynote without reproduction.

</details>


### Lecture 06 · How to Track the Frontier

Develop research literacy, evidence hierarchies, reproduction habits, forecasting, and a review cadence for a fast-moving field.

#### A frontier benchmark can miss the archive

A new model wins a public image benchmark and fails on faded handwritten entries from one exhibit. Elise needs a decision for that collection.

Add a small labeled set from the actual archive with a process for checking errors and coverage. Record format, model version, and what reviewers had to correct before deciding which workflow to deploy.

#### Read the mechanism

**primary sources**

Primary sources are original papers, specs, and official docs—not summaries or hype threads—for technical claims.

Read Attention Is All You Need for architecture claims, not a blog recap.

**benchmarks**

Benchmarks standardize task comparisons—MMLU, HumanEval, BEIR—but may not reflect your production distribution.

High MMLU does not guarantee payroll policy QA performance.

**ablations**

Ablations remove components to measure contribution—essential for judging which mechanism drives reported gains.

Paper claims graph RAG helps; ablation removing graph should show drop if claim holds.

**reproduction**

Reproduction reruns experiments with disclosed details to verify claims before betting architecture on results.

Reproduce reported recall gain within 2 points using authors' config or document differences.

**technology forecasting**

Technology forecasting estimates when emerging capabilities become production-ready using evidence tiers and uncertainty bounds.

Estimate computer-use reliability for your UI stack as low/med/high with dated reassessment.

#### Worked example

**Situation:** Leadership overwhelmed by weekly AI announcements; needs durable process to assess claims affecting onboarding roadmap.

**Baseline:** Adopt every trending technique immediately.

**Application:** Write one-page frontier assessment: primary source, benchmark limits, ablations, reproduction plan, confidence level, mapping to book principles, review cadence.

**Test cases:** (1) Normal: peer-reviewed reproducible result. (2) Boundary: strong benchmark, weak real-world slice. (3) Adversarial: vendor-funded eval with hidden prompt tuning.

**Measurement:** Time to produce assessment, prediction accuracy of adopted vs deferred choices at 6 months.

**Design question:** What confidence level triggers a paid pilot versus continued monitoring?

**Predict before running.** The durable skill is evaluating claims and mapping new mechanisms to established principles. Identify the input you would hold fixed and the observation that would change your decision.


In [ ]:
"""Lab 13.6: How to Track the Frontier"""

assessment = {
    "claim": "new agent framework 2x faster",
    "evidence": "vendor blog",
    "reproduced": False,
    "confidence": "low",
}
action = "monitor" if assessment["confidence"] == "low" else "pilot"
print({"action": action, **assessment})


#### Interpret and apply

Write a one-page frontier assessment with confidence levels.

Change one input, record the prediction before running again, and compare with the unchanged output. If a result disappoints, record which intermediate stage first diverged.

1. What durable skill survives rapid AI change?
2. How do ablations strengthen evidence?
3. What tracking baseline follows hype cycles?

<details><summary>Check your reasoning</summary>

Q1: Evaluating claims and mapping to principles. Q2: Show which component drives gains—not just headline number. Q3: Rewrite stack every launch week.

</details>


**Section checkpoint.** Write a falsifiable claim about a new capability and define the baseline and evidence needed.


## Course workshop

Inspect provenance and the confidence gate on the bundled evidence. Propose a separate review rule for records where confidence is unavailable.

Run the reference implementation, then run the checks below it.


In [ ]:
"""Preserve provenance while combining extracted multimodal evidence."""
from dataclasses import dataclass


@dataclass(frozen=True)
class Evidence:
    content: str
    source: str
    page: int
    modality: str
    confidence: float


items = [
    Evidence("Revenue: 12M", "report.pdf", 3, "table", .98),
    Evidence("Revenue increased", "report.pdf", 4, "chart", .82),
]
usable = [item for item in items if item.confidence >= .90]
for item in usable:
    print(f"{item.content} [{item.source}#page={item.page}; modality={item.modality}]")


### Verify and extend

Run the assertions. Then change one assumption and add a check for the predicted behavior.


In [ ]:
assert all(item.source and item.page > 0 for item in items)
assert len(usable) == 1
assert usable[0].source == "report.pdf" and usable[0].modality == "table"
print('Workshop checks passed')


## Course project

Build a multimodal document system and reproduce one frontier claim against a strong baseline.

Keep your baseline, results, and one failure case so someone else can review the decision you made.
